# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub

In [15]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [16]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [17]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 100
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,<NA>


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Pages with high impressions but low clicks or poor average position should be prioritized for content review and refresh because improving them is more likely to increase traffic.

LOW_CTR
LOW_CLICKS
POOR_POSITION
REFRESH_OPPORTUNITY

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
""").df()


features["ctr"] = (
    features["gsc_clicks"] / features["gsc_impressions"]
).fillna(0)
features["score"] = 0

features.loc[
    (features["gsc_impressions"] >= 100) &
    (features["ctr"] < 0.02),
    "score"
] = 2

features.loc[
    (features["gsc_avg_position"] > 10),
    "score"
] += 1
features["reason_code"] = "NO_ACTION"

features.loc[
    (features["gsc_impressions"] > 100) &
    (features["ctr"] < 0.02),
    "reason_code"
] = "LOW_CTR_HIGH_IMPRESSIONS"

features.loc[
    (features["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_POSITION"
features["action"] = "No Action"

features.loc[
    features["reason_code"] == "LOW_CTR_HIGH_IMPRESSIONS",
    "action"
] = "Refresh Content"

features.loc[
    features["reason_code"] == "LOW_POSITION",
    "action"
] = "Improve SEO"


features.head(20)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.000000,0,NO_ACTION,No Action
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000000,0,NO_ACTION,No Action
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.008000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.000000,0,NO_ACTION,No Action
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.000000,0,NO_ACTION,No Action
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,0.004184,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,0.000000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,0.000000,0,NO_ACTION,No Action
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,0.000000,0,NO_ACTION,No Action
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,0.000000,0,NO_ACTION,No Action


In [19]:
import os


ranked_queue = features.sort_values(
    by="score",
    ascending=False
)
os.makedirs("work/outputs", exist_ok=True)


ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

ranked_queue.head(20)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
628590,2026-03-08,client_c182d11e4862a37d,content_5fa9253a106470ce,193,0,22.948187,0.000000,3,LOW_POSITION,Improve SEO
3610965,2026-03-31,client_20259bd6705d81d4,content_7bc93291f9a5cce1,375,2,36.442667,0.005333,3,LOW_POSITION,Improve SEO
3610996,2026-03-31,client_20259bd6705d81d4,content_086c46f7571e2187,1146,0,36.801920,0.000000,3,LOW_POSITION,Improve SEO
3610994,2026-03-31,client_20259bd6705d81d4,content_31e750b96e6abc08,186,0,25.844086,0.000000,3,LOW_POSITION,Improve SEO
3610993,2026-03-31,client_20259bd6705d81d4,content_e5da37536b39473b,239,3,13.757322,0.012552,3,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3610991,2026-03-31,client_20259bd6705d81d4,content_df6b778e8979533a,369,1,28.972900,0.002710,3,LOW_POSITION,Improve SEO
3610989,2026-03-31,client_20259bd6705d81d4,content_cd510f4f32fddb54,477,0,15.287212,0.000000,3,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3610981,2026-03-31,client_20259bd6705d81d4,content_4d91140d1970b9df,212,0,31.334906,0.000000,3,LOW_POSITION,Improve SEO
3611010,2026-03-31,client_20259bd6705d81d4,content_95f116a73c6b056b,129,0,19.736434,0.000000,3,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3611005,2026-03-31,client_20259bd6705d81d4,content_91190507da748282,1078,4,35.346939,0.003711,3,LOW_POSITION,Improve SEO


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Row 1

Action: Improve SEO

Reason: High impressions with poor average position.

Confidence: High.

What would make it wrong?

If this page already ranks well for its target keywords or if impressions are driven by irrelevant queries.

Row 2

Action: Improve SEO

Reason: High impressions and average position above 20.

Confidence: High.

What would make it wrong?

If ranking cannot be improved because of very strong competition.

Row 3

Action: Improve SEO

Reason: Large number of impressions but poor ranking.

Confidence: High.

What would make it wrong?

If the page is already being updated or intentionally targets low-priority keywords.

Row 4

Action: Refresh Content

Reason: High impressions but very low CTR.

Confidence: Medium.

What would make it wrong?

If the title and meta description are already optimized and the low CTR is caused by search intent.

Row 5

Action: Improve SEO

Reason: Average position is very low.

Confidence: High.

What would make it wrong?

If backlinks or technical SEO issues prevent ranking improvements.

Row 6  
Action: Refresh Content

Why it's here:
This page has many impressions but no clicks. Users see it in search results but are not clicking it.

Confidence:
Medium

What would make it wrong?
If the search intent has changed or the title is already optimized.

Row 7

Action: Refresh Content

Why it's here:
The page ranks well but receives no clicks despite many impressions.

Confidence:
High

What would make it wrong?
If the keyword is informational and naturally receives low CTR.

Row 8
Action: Refresh Content

Why it's here:
The page receives impressions but CTR is still low.

Confidence:
Medium

What would make it wrong?
If competitors have much stronger titles or rich snippets.

Row 9
Action: Improve SEO

Why it's here:
The average ranking is very low, making it difficult for users to find the page.

Confidence:
High

What would make it wrong?
If the page targets a very competitive keyword.

Row 10  
Action: Improve SEO

Why it's here:
The page has a good number of impressions but ranks on later search pages.

Confidence:
Medium

What would make it wrong?
If the keyword is seasonal or search demand recently changed.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may have high impressions because of seasonal events, so they may not actually need a content refresh.

Some pages have low CTR because users already found the answer in the search results, not because the content is poor.

Some pages may already be improving, so refreshing them may not be necessary.

## Leakage Check

No future information was used.

No product flags were included in the rule.

The baseline score only uses:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- CTR

These features are available before making the decision, so there is no data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.